In [36]:
import pandas as pd
import numpy as np
# from tqdm import tqdm
import random
import time
import csv
import json
import openai
import os
from tqdm import tqdm
import time
import ast
import re

#### methods to detect errors in predictions
* *number_of_errors_non_dict:* see how many predictions cannot be transformed to dictionaries
* *fix_non_dict:* transform these instances into dictionaries
* *number_of_errors_dict:* see how many predictions (dict) do not follow the key requirements (ordered set of values)
* *fix_dict*: fix errors
* *is_valid*: see, for each prediction, if follows the requirements, 54 values in order and 1/0 as predictions. 
* *invalid_predictions_format*: see how many predictions are not valid (is_valid)



In [37]:
expected_keys_in_order = [
    'Be creative', 'Be curious', 'Have freedom of thought', 'Be choosing own goals',
    'Be independent', 'Have freedom of action', 'Have privacy', 'Have an exciting life',
    'Have a varied life', 'Be daring', 'Have pleasure', 'Be ambitious', 'Have success',
    'Be capable', 'Be intellectual', 'Be courageous', 'Have influence',
    'Have the right to command', 'Have wealth', 'Have social recognition',
    'Have a good reputation', 'Have a sense of belonging', 'Have good health',
    'Have no debts', 'Be neat and tidy', 'Have a comfortable life',
    'Have a safe country', 'Have a stable society', 'Be respecting traditions',
    'Be holding religious faith', 'Be compliant', 'Be self-disciplined',
    'Be behaving properly', 'Be polite', 'Be honoring elders', 'Be humble',
    'Have life accepted as is', 'Be helpful', 'Be honest', 'Be forgiving',
    'Have the own family secured', 'Be loving', 'Be responsible',
    'Have loyalty towards friends', 'Have equality', 'Be just',
    'Have a world at peace', 'Be protecting the environment',
    'Have harmony with nature', 'Have a world of beauty', 'Be broadminded',
    'Have the wisdom to accept others', 'Be logical', 'Have an objective view'
]
def number_of_errors_non_dict(csvname):
    df = pd.read_csv(csvname) #change name to baseline_gpt_oss_1000
    n_errors, args_ids = 0, []
    for idx, row in df.iterrows():
        try:
            df.at[idx, 'values'] = ast.literal_eval(row['values'])
        except SyntaxError: #string cannot be converted to dictionary (lack of format)
            print(f"Error .to_dict: {row['Argument ID']} -", row['values'])
            args_ids.append(row["Argument ID"])
            n_errors+=1
            pass
        except ValueError: 
            print(f"VALUE ERROR .to_dict: {row['Argument ID']} -", row['values'])
            args_ids.append(row["Argument ID"])
            n_errors+=1

    print(f"\nTotal of predictions that cannot be transformed into a dict type: {n_errors}.\n")
    # type(co_results['values'][0])
    return df, args_ids

def fix_non_dict(
                    df: pd.DataFrame,
                    args_ids: list,
                    expected_keys= list
                ) -> pd.DataFrame: # select those rows that cannot be transformed to dictionaries

    df_fixed = df.copy()

    mask = df_fixed["Argument ID"].isin(args_ids)
    changes_n = 0
    for idx in df_fixed.index[mask]:
        # Save the original string
        values_in_str = str(df_fixed.at[idx, "values"])

        # Extract the numbers
        numbers = [int(x) for x in re.findall(r':\s*([01])(?=,|\})', values_in_str)]
        # print(numbers)
        # Build the dictionary
        new_dict = {key: v for key, v in zip(expected_keys, numbers)}
        # print(new_dict)
        # Replace the value
        df_fixed.at[idx, "values"] = new_dict
        changes_n+=1

    print(df_fixed.at[idx, "Argument ID"],":", df_fixed.loc[idx, "values"])
    print(f"\nTotal of errors solved regarding strings that couldnt be transformed into a dict type: {changes_n}.\n")

    return df_fixed

def number_of_errors_dict(df,expected_keys_in_order):
    n_errors, args_ids = 0, []
    new_errors = 0
    for (idx,value), arg_id in zip(df["values"].items(),df['Argument ID']):
        if isinstance(value, dict):
            if list(value.keys()) != expected_keys_in_order:
                new_errors += 1
                args_ids.append(arg_id)
            else:
                pass
        else:
            n_errors+=1
            # args_ids.append(arg_id)
            print(f"\nArgument ID {arg_id}: False ({type(value).__name__})")
            print(value)

    print(f"\nTotal of predictions that are still str type and have not been transformed to dict: {n_errors}.\n")
    print(f"Total of predictions that do not follow the same number and values as the reference: {new_errors}.\n")
    return args_ids
    
def fix_dict(
                    df: pd.DataFrame,
                    args_ids: list,
                    expected_keys= list
                ) -> pd.DataFrame: # select those rows that cannot be transformed to dictionaries

    df_fixed = df.copy()

    mask = df_fixed["Argument ID"].isin(args_ids)
    changes_n = 0
    for idx in df_fixed.index[mask]:
        dictionary_output = df_fixed.at[idx, "values"]
        int_list = list(dictionary_output.values())
        new_dict = {key: v for key, v in zip(expected_keys, int_list)}
        df_fixed.at[idx, "values"] = new_dict
        changes_n+=1
        print(df_fixed.at[idx, "Argument ID"],":", df_fixed.loc[idx, "values"])
    print(f"\nTotal of errors solved regarding strings that couldnt be transformed into a dict type: {changes_n}.\n")

    return df_fixed

def invalid_predictions_format(df):
    def is_valid(d):
        return (
            isinstance(d, dict)
            and len(d) == 54 # se han predicho 54 valores
            and list(d.keys()) == expected_keys_in_order # se cumple el orden y los nombres de los valores presentes en la lista original del prompt
            and all(isinstance(v, (int)) for v in d.values()) # es un número entero
            and all(v in (0, 1) for v in d.values()) # los valores son 0 o 1
        )

    valid_mask = df["values"].apply(is_valid)

    all_valid = valid_mask.all()

    invalid_rows = df[~valid_mask]

    print(len(invalid_rows), "arguments have not been predicted according to the specified rules.")

    for arg, bad_prediction in zip(invalid_rows["Argument ID"],invalid_rows["values"]):
        print("\n\nArg:", arg, "no cumple los requisitos de clasificación:\n")
        # print(bad_prediction)
        try:
            print("Has 54 values predicted", len(bad_prediction) == 54)
            print("54 values keys are ordered", list(bad_prediction.keys()) == expected_keys_in_order)
            print("All predictions are integers",all(isinstance(v, (int)) for v in bad_prediction.values())) # es un número entero
            print("All predictions are either 0 or 1",all(v in (0, 1) for v in bad_prediction.values())) # los valores son 0 o 1

            dict_keys = set(bad_prediction.keys())
            expected_keys = set(expected_keys_in_order)

            missing = expected_keys - dict_keys
            extra = dict_keys - expected_keys
            print("The error detected is:", extra)
            print()
            print()
        except AttributeError:
            print()
            print("Attribute error",bad_prediction)
    
    invalid_ids = df.loc[~valid_mask, "Argument ID"].tolist()

    return invalid_ids

## Baseline - LLM predictor without bias nor heuristics


#### Error Checking in first prediction of values in  baseline


Check if...
- Strings can be converted into a dictionary
- The predictions have been made correctly (dictionary with 54 keys and 54 values that are integers 0 / 1)

In [38]:
# TRANSFORM THE LLMS' OUTPUT INTO A REAL DICTIONARY and check errors baseline_results
baseline = "results/baseline_gpt_oss_1000.csv"
baseline_df, erroneous_ids_non_dict = number_of_errors_non_dict(baseline) #change name to baseline_gpt_oss_1000
fixed_df = fix_non_dict(baseline_df, erroneous_ids_non_dict, expected_keys_in_order)
print("The error has been solved? dict -> Yes")
type(fixed_df.loc[fixed_df["Argument ID"] == "A23244", "values"].iloc[0]) # the error has been fixed, now it is a dictionary

Error .to_dict: A23244 - {'Be creative - allowing for more creativity or imagination; being more creative; fostering creativity; promoting imagination':0, 'Be curious - being the more interesting option; fostering curiosity; making people more keen to learn; promoting discoveries; sparking interest':0, 'Have freedom of thought - allowing people to figure things out on their own; allowing people to make up their mind; resulting in less censorship; resulting in less influence on people's thoughts':1, 'Be choosing own goals - allowing people to choose what is best for them; allowing people to decide on their life; allowing people to follow their dreams':1, 'Be independent - allowing people to plan on their own; resulting in fewer times people have to ask for consent':1, 'Have freedom of action - allowing people to be self-determined; allowing people to do things even though this may hurt them in the long run; resulting in more times people can do what they want':1, 'Have privacy - allowin

dict

In [39]:
erroneous_ids_dict = number_of_errors_dict(fixed_df, expected_keys_in_order)
fixed_df = fix_dict(fixed_df,erroneous_ids_dict, expected_keys_in_order)


Total of predictions that are still str type and have not been transformed to dict: 0.

Total of predictions that do not follow the same number and values as the reference: 43.

A01006 : {'Be creative': 0, 'Be curious': 0, 'Have freedom of thought': 0, 'Be choosing own goals': 0, 'Be independent': 0, 'Have freedom of action': 0, 'Have privacy': 0, 'Have an exciting life': 0, 'Have a varied life': 0, 'Be daring': 0, 'Have pleasure': 0, 'Be ambitious': 0, 'Have success': 0, 'Be capable': 0, 'Be intellectual': 0, 'Be courageous': 1, 'Have influence': 1, 'Have the right to command': 0, 'Have wealth': 0, 'Have social recognition': 0, 'Have a good reputation': 0, 'Have a sense of belonging': 0, 'Have good health': 0, 'Have no debts': 0, 'Be neat and tidy': 0, 'Have a comfortable life': 0, 'Have a safe country': 0, 'Have a stable society': 1, 'Be respecting traditions': 0, 'Be holding religious faith': 0, 'Be compliant': 0, 'Be self-disciplined': 0, 'Be behaving properly': 0, 'Be polite': 0,

In [40]:
invalid_predictions_format(fixed_df)

0 arguments have not been predicted according to the specified rules.


[]

In [41]:
fixed_df.to_csv('valid_results/baseline_1000.csv', index=False)

## Heuristic solving errors

In [42]:
bias = "heuristics"
csvname = f"results/{bias}_gpt_oss_503.csv"
baseline_df, erroneous_ids_non_dict = number_of_errors_non_dict(csvname) #change name to baseline_gpt_oss_1000
fixed_df = fix_non_dict(baseline_df, erroneous_ids_non_dict, expected_keys_in_order)
print("The error has been solved? dict -> Yes")
# type(fixed_df.loc[fixed_df["Argument ID"] == "A23244", "values"].iloc[0])

Error .to_dict: A12077 - {'Be creative - allowing for more creativity or imagination; being more creative; fostering creativity; promoting imagination':1, 'Be curious - being the more interesting option; fostering curiosity; making people more keen to learn; promoting discoveries; sparking interest':0, 'Have freedom of thought - allowing people to figure things out on their own; allowing people to make up their mind; resulting in less censorship; resulting in less influence on people's thoughts':0, 'Be choosing own goals - allowing people to choose what is best for them; allowing people to decide on their life; allowing people to follow their dreams':0, 'Be independent - allowing people to plan on their own; resulting in fewer times people have to ask for consent':1, 'Have freedom of action - allowing people to be self-determined; allowing people to do things even though this may hurt them in the long run; resulting in more times people can do what they want':0, 'Have privacy - allowin

In [43]:
erroneous_ids_dict = number_of_errors_dict(fixed_df, expected_keys_in_order)
fixed_df = fix_dict(fixed_df,erroneous_ids_dict, expected_keys_in_order)


Total of predictions that are still str type and have not been transformed to dict: 0.

Total of predictions that do not follow the same number and values as the reference: 24.

A07017 : {'Be creative': 0, 'Be curious': 0, 'Have freedom of thought': 0, 'Be choosing own goals': 0, 'Be independent': 0, 'Have freedom of action': 0, 'Have privacy': 0, 'Have an exciting life': 0, 'Have a varied life': 0, 'Be daring': 0, 'Have pleasure': 0, 'Be ambitious': 0, 'Have success': 0, 'Be capable': 0, 'Be intellectual': 0, 'Be courageous': 0, 'Have influence': 0, 'Have the right to command': 0, 'Have wealth': 0, 'Have social recognition': 0, 'Have a good reputation': 0, 'Have a sense of belonging': 0, 'Have good health': 1, 'Have no debts': 0, 'Be neat and tidy': 0, 'Have a comfortable life': 0, 'Have a safe country': 0, 'Have a stable society': 0, 'Be respecting traditions': 1, 'Be holding religious faith': 0, 'Be compliant': 0, 'Be self-disciplined': 0, 'Be behaving properly': 0, 'Be polite': 0,

In [44]:
invalid_predictions_format(fixed_df)
fixed_df.to_csv(f'valid_results/{bias}_503.csv', index=False)

0

 arguments have not been predicted according to the specified rules.


## Bias solving errors

### Confirmation Bias

In [45]:
bias = "confirmation_bias"
csvname = f"results/{bias}_gpt_oss_503.csv"
baseline_df, erroneous_ids_non_dict = number_of_errors_non_dict(csvname) #change name to baseline_gpt_oss_1000
fixed_df = fix_non_dict(baseline_df, erroneous_ids_non_dict, expected_keys_in_order)
print("The error has been solved? dict -> Yes")
type(fixed_df.loc[fixed_df["Argument ID"] == "A23244", "values"].iloc[0]) # the error has been fixed, now it is a dictionary

Error .to_dict: A06017 - {'Be creative - allowing for more creativity or imagination; being more creative; fostering creativity; promoting imagination':0, 'Be curious - being the more interesting option; fostering curiosity; making people more keen to learn; promoting discoveries; sparking interest':0, 'Have freedom of thought - allowing people to figure things out on their own; allowing people to make up their mind; resulting in less censorship; resulting in less influence on people's thoughts':0, 'Be choosing own goals - allowing people to choose what is best for them; allowing people to decide on their life; allowing people to follow their dreams':0, 'Be independent - allowing people to plan on their own; resulting in fewer times people have to ask for consent':0, 'Have freedom of action - allowing people to be self-determined; allowing people to do things even though this may hurt them in the long run; resulting in more times people can do what they want':0, 'Have privacy - allowin

dict

In [46]:
erroneous_ids_dict = number_of_errors_dict(fixed_df, expected_keys_in_order)
fixed_df = fix_dict(fixed_df,erroneous_ids_dict, expected_keys_in_order)


Total of predictions that are still str type and have not been transformed to dict: 0.

Total of predictions that do not follow the same number and values as the reference: 27.

A05041 : {'Be creative': 1, 'Be curious': 0, 'Have freedom of thought': 1, 'Be choosing own goals': 0, 'Be independent': 1, 'Have freedom of action': 0, 'Have privacy': 0, 'Have an exciting life': 0, 'Have a varied life': 0, 'Be daring': 0, 'Have pleasure': 0, 'Be ambitious': 0, 'Have success': 0, 'Be capable': 0, 'Be intellectual': 1, 'Be courageous': 0, 'Have influence': 0, 'Have the right to command': 0, 'Have wealth': 0, 'Have social recognition': 0, 'Have a good reputation': 0, 'Have a sense of belonging': 0, 'Have good health': 0, 'Have no debts': 0, 'Be neat and tidy': 0, 'Have a comfortable life': 0, 'Have a safe country': 0, 'Have a stable society': 0, 'Be respecting traditions': 0, 'Be holding religious faith': 0, 'Be compliant': 0, 'Be self-disciplined': 0, 'Be behaving properly': 0, 'Be polite': 0,

In [47]:
invalid_predictions_format(fixed_df)

0 arguments have not been predicted according to the specified rules.


[]

In [48]:
fixed_df.to_csv(f'valid_results/{bias}_503.csv', index=False)

### Causal Oversimplification

In [49]:
## Confirmation Bias
bias = "causal_oversimplification"
csvname = f"results/{bias}_gpt_oss_503.csv"
baseline_df, erroneous_ids_non_dict = number_of_errors_non_dict(csvname) #change name to baseline_gpt_oss_1000
fixed_df = fix_non_dict(baseline_df, erroneous_ids_non_dict, expected_keys_in_order)
print("The error has been solved? dict -> Yes")
type(fixed_df.loc[fixed_df["Argument ID"] == "A23244", "values"].iloc[0]) # the error has been fixed, now it is a dictionary
erroneous_ids_dict = number_of_errors_dict(fixed_df, expected_keys_in_order)
fixed_df = fix_dict(fixed_df,erroneous_ids_dict, expected_keys_in_order)
invalid_predictions_format(fixed_df)
fixed_df.to_csv(f'valid_results/{bias}_503.csv', index=False)#

Error .to_dict: A12052 - {'Be creative - allowing for more creativity or imagination; being more creative; fostering creativity; promoting imagination':0, 'Be curious - being the more interesting option; fostering curiosity; making people more keen to learn; promoting discoveries; sparking interest':0, 'Have freedom of thought - allowing people to figure things out on their own; allowing people to make up their mind; resulting in less censorship; resulting in less influence on people's thoughts':0, 'Be choosing own goals - allowing people to choose what is best for them; allowing people to decide on their life; allowing people to follow their dreams':0, 'Be independent - allowing people to plan on their own; resulting in fewer times people have to ask for consent':0, 'Have freedom of action - allowing people to be self-determined; allowing people to do things even though this may hurt them in the long run; resulting in more times people can do what they want':0, 'Have privacy - allowin

### Hasty Generalization

In [50]:
bias = "hasty_generalization"
csvname = f"results/{bias}_gpt_oss_503.csv"
baseline_df, erroneous_ids_non_dict = number_of_errors_non_dict(csvname) #change name to baseline_gpt_oss_1000
fixed_df = fix_non_dict(baseline_df, erroneous_ids_non_dict, expected_keys_in_order)
print("The error has been solved? dict -> Yes")
type(fixed_df.loc[fixed_df["Argument ID"] == "A23244", "values"].iloc[0]) # the error has been fixed, now it is a dictionary
erroneous_ids_dict = number_of_errors_dict(fixed_df, expected_keys_in_order)
fixed_df = fix_dict(fixed_df,erroneous_ids_dict, expected_keys_in_order)
invalid_predictions_format(fixed_df)
fixed_df.to_csv(f'valid_results/{bias}_503.csv', index=False)

Error .to_dict: A05042 - {'Be creative - allowing for more creativity or imagination; being more creative; fostering creativity; promoting imagination':0, 'Be curious - being the more interesting option; fostering curiosity; making people more keen to learn; promoting discoveries; sparking interest':0, 'Have freedom of thought - allowing people to figure things out on their own; allowing people to make up their mind; resulting in less censorship; resulting in less influence on people's thoughts':1, 'Be choosing own goals - allowing people to choose what is best for them; allowing people to decide on their life; allowing people to follow their dreams':1, 'Be independent - allowing people to plan on their own; resulting in fewer times people have to ask for consent':1, 'Have freedom of action - allowing people to be self-determined; allowing people to do things even though this may hurt them in the long run; resulting in more times people can do what they want':0, 'Have privacy - allowin

### Motivated Reasoning
* a single prediction is nan, repeat process (LLM-Call, cannot be solved using regex) Argument ID - VALUE ERROR .to_dict: A19184 - nan

In [51]:
bias = "motivated_reasoning"
csvname = f"results/{bias}_gpt_oss_503.csv"
baseline_df, erroneous_ids_non_dict = number_of_errors_non_dict(csvname) #change name to baseline_gpt_oss_1000
fixed_df = fix_non_dict(baseline_df, erroneous_ids_non_dict, expected_keys_in_order)
print("The error has been solved? dict -> Yes")
type(fixed_df.loc[fixed_df["Argument ID"] == "A23244", "values"].iloc[0]) # the error has been fixed, now it is a dictionary
erroneous_ids_dict = number_of_errors_dict(fixed_df, expected_keys_in_order)
fixed_df = fix_dict(fixed_df,erroneous_ids_dict, expected_keys_in_order)
invalid_predictions_format(fixed_df)
fixed_df.to_csv(f'valid_results/{bias}_503.csv', index=False)

Error .to_dict: A12022 - {'Be creative - allowing for more creativity or imagination; being more creative; fostering creativity; promoting imagination':0, 'Be curious - being the more interesting option; fostering curiosity; making people more keen to learn; promoting discoveries; sparking interest':0, 'Have freedom of thought - allowing people to figure things out on their own; allowing people to make up their mind; resulting in less censorship; resulting in less influence on people's thoughts':1, 'Be choosing own goals - allowing people to choose what is best for them; allowing people to decide on their life; allowing people to follow their dreams':1, 'Be independent - allowing people to plan on their own; resulting in fewer times people have to ask for consent':1, 'Have freedom of action - allowing people to be self-determined; allowing people to do things even though this may hurt them in the long run; resulting in more times people can do what they want':1, 'Have privacy - allowin

### Framing Effect
- One Nan - A05051,None,22.013577222824097

In [52]:
bias = "framing_effect"
csvname = f"results/{bias}_gpt_oss_503.csv"
baseline_df, erroneous_ids_non_dict = number_of_errors_non_dict(csvname) #change name to baseline_gpt_oss_1000
fixed_df = fix_non_dict(baseline_df, erroneous_ids_non_dict, expected_keys_in_order)
print("The error has been solved? dict -> Yes")
type(fixed_df.loc[fixed_df["Argument ID"] == "A23244", "values"].iloc[0]) # the error has been fixed, now it is a dictionary
erroneous_ids_dict = number_of_errors_dict(fixed_df, expected_keys_in_order)
fixed_df = fix_dict(fixed_df,erroneous_ids_dict, expected_keys_in_order)
invalid_predictions_format(fixed_df)
fixed_df.to_csv(f'valid_results/{bias}_503.csv', index=False)

Error .to_dict: A05042 - {"Be creative - allowing for more creativity or imagination; being more creative; fostering creativity; promoting imagination":0, "Be curious - being the more interesting option; fostering curiosity; making people more keen to learn; promoting discoveries; sparking interest":0, "Have freedom of thought - allowing people to figure things out on their own; allowing people to make up their mind; resulting in less censorship; resulting in less influence on people's thoughts":0, "Be choosing own goals - allowing people to choose what is best for them; allowing people to decide on their life; allowing people to follow their dreams":0, "Be independent - allowing people to plan on their own; resulting in fewer times people have to ask for consent":0, "Have freedom of action - allowing people to be self-determined; allowing people to do things even though this may hurt them in the long run; resulting in more times people can do what they want":0, "Have privacy - allowin